# 03 — Embeddings & FAISS (Instructor Notebook)

This notebook explains what embeddings are, why they are useful for semantic search, and provides runnable examples using sentence-transformers (or TF-IDF fallback) and FAISS (or cosine fallback). Each step includes explanations and instructor notes.

## 1) What is an embedding? (what/why)

An embedding is a fixed-length numeric vector representing semantic information about text or sequences. Why: it lets you compare meaning using vector similarity rather than keyword matching.

In [ ]:
# Small dataset for demo
docs = [
    'DNA sequencing determines the order of nucleotides',
    'CRISPR is a gene editing tool',
    'Protein folding depends on amino acid sequence',
    'The dog is sleeping in the park'
]
print('Documents:', len(docs))

## 2) Encode with sentence-transformers or fallback (what/why)

What: Create vector representations. Why: Use vectors to compute similarity instead of string matching.

In [ ]:
# Try to use sentence-transformers, else fall back to TF-IDF
import numpy as np
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(docs)
    print('Used sentence-transformers, embedding dim=', embeddings.shape[1])
    def embed(text):
        return model.encode([text])[0]
except Exception as e:
    print('sentence-transformers not available, using TF-IDF fallback')
    from sklearn.feature_extraction.text import TfidfVectorizer
    v = TfidfVectorizer().fit(docs)
    embeddings = v.transform(docs).toarray()
    def embed(text):
        return v.transform([text]).toarray()[0]

## 3) Build a FAISS index (or fallback) — what/why

What: Build a vector index to allow fast nearest-neighbor search. Why: FAISS scales to millions of vectors; for small demos, exact search is fine.

In [ ]:
# Try FAISS, else do cosine fallback
try:
    import faiss
    d = embeddings.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(embeddings.astype('float32'))
    print('FAISS index built, total:', index.ntotal)
    def search(query, k=3):
        q = embed(query).astype('float32').reshape(1,-1)
        D, I = index.search(q, k)
        return [(docs[i], float(D[0][j])) for j,i in enumerate(I[0])]
except Exception as e:
    print('FAISS not available — using cosine similarity fallback')
    from sklearn.metrics.pairwise import cosine_similarity
    def search(query, k=3):
        q = embed(query).reshape(1,-1)
        sims = cosine_similarity(q, embeddings)[0]
        idx = np.argsort(-sims)[:k]
        return [(docs[i], float(sims[i])) for i in idx]

In [ ]:
# Try a query
query = 'How does CRISPR work?'
print('Query:', query)
results = search(query, k=3)
print('
Top results:')
for r in results:
    print('-', r)

## Exercise B — Build and query index

Task: Change or add documents (e.g., add more biology sentences) and observe how results change. Explain why some documents are ranked higher.
Instructor solution: Documents closely matching the semantic content of the query (CRISPR/gene editing) will rank higher; TF-IDF will prefer exact keyword overlap while embeddings capture semantics.